# Notebook 01 — Phân tích tín hiệu âm thanh (Signal Analysis)
**DSP501 — Nhận diện người nói (Speaker Identification)**

## Mục tiêu
Notebook này phân tích **tín hiệu âm thanh thô** của một file mẫu, sau đó áp dụng **bộ lọc FIR bandpass** và so sánh kết quả trước/sau lọc.

### Nội dung:
1. Load và tiền xử lý audio (normalize, trim silence, pad/crop 3 giây)
2. Thiết kế bộ lọc FIR bandpass 300–3400 Hz
3. Phân tích đáp ứng tần số & đáp ứng pha của bộ lọc
4. So sánh dạng sóng trước/sau lọc
5. So sánh phổ tần số (FFT)
6. So sánh Spectrogram (STFT)
7. Mật độ phổ công suất (PSD)
8. Tỷ số tín hiệu trên nhiễu (SNR)
9. Pre-emphasis — tăng cường tần số cao

### Dataset hiện tại:
- 4 speakers: **Cuong**, **Quang**, **Anne**, **Khoa**
- Mỗi speaker: 25 file WAV (3 giây, 16kHz, mono)
- Tổng: 100 mẫu

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from scipy.signal import welch

from preprocess import preprocess
from filter import design_fir, apply_filter
from preemphasis import pre_emphasize
from analysis import plot_waveform, plot_spectrum, plot_stft, compute_psd, compute_snr

import os
os.makedirs('../figures', exist_ok=True)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

SR = 16000          # Tần số lấy mẫu 16kHz
TARGET_LEN = 48000  # 3 giây × 16000 Hz

print('Import thành công!')

---
## 1. Load và tiền xử lý audio

Mỗi file audio đi qua **4 bước tiền xử lý** trước khi phân tích:

| Bước | Hàm | Mục đích |
|------|-----|----------|
| 1 | `load_audio()` | Đọc file WAV, chuyển mono, resample 16kHz |
| 2 | `normalize()` | Chuẩn hóa biên độ về [-1, 1] để các file có cùng mức âm lượng |
| 3 | `trim_silence()` | Cắt bỏ khoảng lặng đầu/cuối (ngưỡng -20dB) |
| 4 | `pad_or_crop()` | Đảm bảo đúng 48,000 mẫu (3 giây × 16,000 Hz) |

**Tại sao cần tiền xử lý?**
- Các file ghi âm khác nhau có **âm lượng khác nhau** → normalize giúp đồng nhất
- Đầu/cuối file thường có **khoảng lặng** → cắt bỏ để tập trung vào giọng nói
- SVM yêu cầu vector đầu vào **cùng kích thước** → pad/crop về 3 giây

Ta dùng file của **Cuong** (`speaker_08/01.wav`) làm ví dụ.

In [ ]:
# Load 1 file mẫu của Cuong
SAMPLE = '../data/raw/speaker_08/01.wav'

y_raw, sr = preprocess(SAMPLE, sr=SR, target_len=TARGET_LEN)

print('=== THÔNG TIN FILE AUDIO ===')
print(f'  File          : {SAMPLE}')
print(f'  Tần số lấy mẫu: {sr} Hz')
print(f'  Số mẫu        : {len(y_raw):,} samples')
print(f'  Thời lượng    : {len(y_raw)/sr:.1f} giây')
print(f'  Biên độ       : [{y_raw.min():.3f}, {y_raw.max():.3f}]')
print(f'  RMS Energy    : {np.sqrt(np.mean(y_raw**2)):.4f}')

# Vẽ dạng sóng
time = np.arange(len(y_raw)) / sr
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(time, y_raw, linewidth=0.5, color='steelblue')
ax.set_xlabel('Thời gian (giây)')
ax.set_ylabel('Biên độ')
ax.set_title('Dạng sóng sau tiền xử lý — Cuong (speaker_08/01.wav)')
ax.set_xlim(0, len(y_raw)/sr)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nĐọc biểu đồ:')
print('  - Trục X: thời gian (giây) — file dài đúng 3 giây')
print('  - Trục Y: biên độ [-1, 1] — đã normalize')
print('  - Phần có sóng lớn = giọng nói, phần phẳng = im lặng')

---
## 2. Thiết kế bộ lọc FIR Bandpass (300–3400 Hz)

### Bộ lọc FIR (Finite Impulse Response) là gì?
- Bộ lọc số chỉ dùng **đầu vào hiện tại và quá khứ** (không phản hồi)
- Luôn **ổn định** (stable) — không bao giờ phát tán
- Với hệ số đối xứng → **pha tuyến tính** (không méo dạng sóng)

### Tại sao chọn 300–3400 Hz?
Đây là dải **telephone speech band** (tiêu chuẩn ITU-T G.711):

| Dải tần | Nội dung | Xử lý |
|---------|----------|-------|
| **< 300 Hz** | Tiếng ồn nền, hum điện 50Hz, rung động | **Loại bỏ** |
| **300–3400 Hz** | Formants F1, F2, F3 — đặc trưng giọng nói | **Giữ lại** |
| **> 3400 Hz** | Nhiễu tần số cao, tiếng xì, ít thông tin | **Loại bỏ** |

### Thông số bộ lọc:
- **Số tap: 101** — bậc lọc cao → dốc sắc → lọc chính xác
- **Cửa sổ: Hamming** — side lobe < -40dB, kiểm soát rò rỉ tốt
- **Loại: FIR** (không phải IIR) — vì cần pha tuyến tính cho MFCC

In [ ]:
# Thiết kế bộ lọc FIR bandpass
coeffs = design_fir(lowcut=300, highcut=3400, sr=SR, numtaps=101)

print('=== THÔNG SỐ BỘ LỌC FIR ===')
print(f'  Loại        : Bandpass (lọc dải thông)')
print(f'  Dải thông   : 300 – 3400 Hz')
print(f'  Số tap      : {len(coeffs)} (bậc lọc = {len(coeffs) - 1})')
print(f'  Cửa sổ      : Hamming')
print(f'  Pha         : Tuyến tính (hệ số đối xứng)')
print(f'\n  5 hệ số đầu tiên: {coeffs[:5].round(6)}')
print(f'  Hệ số giữa (lớn nhất): {coeffs[50]:.6f}')

# Kiểm tra đối xứng
is_symmetric = np.allclose(coeffs, coeffs[::-1])
print(f'  Đối xứng?   : {is_symmetric} → pha tuyến tính được đảm bảo')

---
## 3. Đáp ứng tần số & Đáp ứng pha của bộ lọc

### Đáp ứng tần số (Frequency Response):
- Trục X: tần số (Hz)
- Trục Y: biên độ (dB)
- **Dải thông (passband)**: 300–3400 Hz → biên độ ≈ 0 dB (giữ nguyên)
- **Dải chắn (stopband)**: ngoài 300–3400 Hz → biên độ < -40 dB (suy giảm mạnh)

### Đáp ứng pha (Phase Response):
- Pha **tuyến tính** trong dải thông → tín hiệu không bị méo dạng
- Quan trọng cho MFCC vì FFT trong MFCC nhạy cảm với pha

### Đáp ứng xung (Impulse Response):
- Chính là **các hệ số bộ lọc** — vì FIR không có phản hồi
- Đối xứng qua điểm giữa → đặc trưng Type I FIR → pha tuyến tính

In [ ]:
from scipy.signal import freqz

# Tính đáp ứng tần số
w, h = freqz(coeffs, worN=2048)
freqs_hz = w * SR / (2 * np.pi)
mag_db = 20 * np.log10(np.abs(h) + 1e-10)
phase_rad = np.unwrap(np.angle(h))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Đáp ứng tần số (biên độ)
axes[0].plot(freqs_hz, mag_db, color='blue', linewidth=1.5)
axes[0].axvline(300, color='red', linestyle='--', alpha=0.8, label='300 Hz')
axes[0].axvline(3400, color='green', linestyle='--', alpha=0.8, label='3400 Hz')
axes[0].axhline(-3, color='gray', linestyle=':', alpha=0.5, label='-3 dB')
axes[0].set_title('Đáp ứng tần số (Magnitude)')
axes[0].set_xlabel('Tần số (Hz)')
axes[0].set_ylabel('Biên độ (dB)')
axes[0].set_xlim(0, SR/2)
axes[0].set_ylim(-80, 5)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# 2. Đáp ứng pha
axes[1].plot(freqs_hz, phase_rad, color='green', linewidth=1.5)
axes[1].axvline(300, color='red', linestyle='--', alpha=0.8)
axes[1].axvline(3400, color='green', linestyle='--', alpha=0.8)
axes[1].set_title('Đáp ứng pha (Phase)')
axes[1].set_xlabel('Tần số (Hz)')
axes[1].set_ylabel('Pha (radian)')
axes[1].set_xlim(0, SR/2)
axes[1].grid(True, alpha=0.3)

# 3. Đáp ứng xung (hệ số bộ lọc)
axes[2].stem(np.arange(len(coeffs)), coeffs, basefmt=' ', markerfmt='o', linefmt='C0-')
axes[2].set_title('Đáp ứng xung (Impulse Response)')
axes[2].set_xlabel('Chỉ số hệ số (n)')
axes[2].set_ylabel('Biên độ')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Phân tích bộ lọc FIR Bandpass 300–3400 Hz', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/fir_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Đọc biểu đồ:')
print('  1. Magnitude: Dải 300–3400 Hz ở ~0dB (giữ nguyên), ngoài dải < -40dB (loại bỏ)')
print('  2. Phase: Đường thẳng trong dải thông → pha tuyến tính → không méo tín hiệu')
print('  3. Impulse: Đối xứng qua điểm giữa (n=50) → đảm bảo pha tuyến tính')

---
## 4. Áp dụng bộ lọc & So sánh dạng sóng

Bây giờ ta áp dụng bộ lọc FIR lên tín hiệu thô và so sánh **dạng sóng trước/sau lọc**.

**Kỳ vọng:**
- Hình dạng tổng thể **tương tự** (giọng nói nằm trong dải 300–3400 Hz)
- Biên độ có thể **giảm nhẹ** (vì đã loại một phần năng lượng)
- Phần lặng **phẳng hơn** (nhiễu nền bị loại)

In [ ]:
# Áp dụng bộ lọc FIR
y_filt = apply_filter(y_raw, coeffs)

print(f'Trước lọc — Biên độ: [{y_raw.min():.3f}, {y_raw.max():.3f}], RMS: {np.sqrt(np.mean(y_raw**2)):.4f}')
print(f'Sau lọc   — Biên độ: [{y_filt.min():.3f}, {y_filt.max():.3f}], RMS: {np.sqrt(np.mean(y_filt**2)):.4f}')

# So sánh dạng sóng
plot_waveform(y_raw, y_filt, sr=SR, save_path='../figures/waveform_comparison.png')

# Zoom vào 0.5 giây đầu để thấy chi tiết
n_show = int(0.5 * SR)  # 0.5 giây
t_show = np.arange(n_show) / SR

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_show, y_raw[:n_show], linewidth=0.8, color='steelblue')
axes[0].set_ylabel('Biên độ')
axes[0].set_title('Zoom 0.5 giây — Tín hiệu thô (Raw)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_show, y_filt[:n_show], linewidth=0.8, color='darkorange')
axes[1].set_ylabel('Biên độ')
axes[1].set_xlabel('Thời gian (giây)')
axes[1].set_title('Zoom 0.5 giây — Sau lọc FIR 300–3400 Hz')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nNhận xét:')
print('  - Dạng sóng tổng thể tương tự → giọng nói nằm trong dải 300–3400 Hz')
print('  - Tín hiệu sau lọc "mượt" hơn → thành phần nhiễu tần số cao đã bị loại')
print('  - Phần im lặng phẳng hơn → nhiễu nền bị suy giảm')

---
## 5. So sánh phổ tần số (FFT)

### FFT (Fast Fourier Transform) là gì?
- Biến đổi tín hiệu từ **miền thời gian** (biên độ vs thời gian) sang **miền tần số** (biên độ vs tần số)
- Cho biết **năng lượng** tín hiệu ở mỗi tần số

**Cách đọc biểu đồ:**
- Trục X: tần số (Hz)
- Trục Y: biên độ (dB) — thang logarithm
- Đường **xanh** (Raw): có năng lượng ở mọi tần số
- Đường **cam** (Filtered): năng lượng chỉ tập trung trong 300–3400 Hz
- 2 đường đỏ/xanh lá: ranh giới bộ lọc (300 Hz và 3400 Hz)

In [ ]:
# So sánh phổ FFT trước/sau lọc
plot_spectrum(y_raw, y_filt, sr=SR, save_path='../figures/spectrum_comparison.png')

print('Nhận xét:')
print('  - Đường cam (sau lọc) suy giảm mạnh < 300 Hz và > 3400 Hz')
print('  - Trong dải 300–3400 Hz, 2 đường gần trùng nhau → giữ nguyên thông tin giọng nói')
print('  - Phần bị cắt chủ yếu là nhiễu → lọc không mất thông tin quan trọng')

---
## 6. So sánh Spectrogram (STFT)

### Spectrogram là gì?
FFT chỉ cho thấy **tổng năng lượng** ở mỗi tần số. Spectrogram cho thấy **tần số nào xuất hiện ở thời điểm nào**.

**Cách tính:** Chia tín hiệu thành frames nhỏ (32ms) → FFT từng frame → xếp cạnh nhau.

**Cách đọc:**
- Trục X: thời gian (giây)
- Trục Y: tần số (Hz)
- Màu sắc: năng lượng (dB) — **vàng/sáng = năng lượng cao**, **tím/tối = năng lượng thấp**
- Các **vạch ngang sáng** = formants (đặc trưng giọng nói của mỗi người)

In [ ]:
# So sánh spectrogram trước/sau lọc
plot_stft(y_raw, y_filt, sr=SR, save_path='../figures/stft_comparison.png')

print('Nhận xét:')
print('  - Raw: năng lượng trải đều từ 0 → 8000 Hz')
print('  - Filtered: năng lượng tập trung trong 300–3400 Hz')
print('  - Vùng < 300 Hz và > 3400 Hz tối hẳn → nhiễu đã bị loại')
print('  - Formants (vạch ngang sáng) vẫn rõ ràng trong dải thông')

---
## 7. Mật độ phổ công suất (PSD — Power Spectral Density)

### PSD là gì?
- Giống FFT nhưng **ổn định hơn** — dùng phương pháp **Welch** (trung bình nhiều đoạn FFT)
- Cho biết **công suất trung bình** ở mỗi tần số
- Đơn vị: V²/Hz (power per Hz)

**Tại sao dùng PSD thay vì FFT?**
- FFT của 1 lần đo bị **dao động nhiều** (noisy)
- PSD chia tín hiệu thành nhiều đoạn, FFT từng đoạn rồi **lấy trung bình** → mượt hơn

**Cách đọc:**
- Thang Y logarithm (semilogy)
- Đường cam thấp hơn đường xanh ở ngoài dải 300–3400 Hz → bộ lọc hoạt động tốt

In [ ]:
# Tính PSD bằng Welch's method
freqs_r, psd_r = compute_psd(y_raw, SR)
freqs_f, psd_f = compute_psd(y_filt, SR)

plt.figure(figsize=(10, 4))
plt.semilogy(freqs_r, psd_r, label='Raw (thô)', alpha=0.8, color='steelblue')
plt.semilogy(freqs_f, psd_f, label='Filtered (sau lọc)', alpha=0.8, color='darkorange')
plt.axvline(300, color='red', linestyle='--', linewidth=0.8, label='300 Hz')
plt.axvline(3400, color='green', linestyle='--', linewidth=0.8, label='3400 Hz')
plt.axvspan(0, 300, alpha=0.1, color='red', label='Dải bị loại')
plt.axvspan(3400, SR/2, alpha=0.1, color='red')
plt.title('Mật độ phổ công suất (PSD) — Welch\'s method')
plt.xlabel('Tần số (Hz)')
plt.ylabel('PSD (V²/Hz)')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/psd_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Nhận xét:')
print('  - Trong dải 300–3400 Hz: 2 đường gần trùng → giữ nguyên công suất giọng nói')
print('  - Ngoài dải: đường cam thấp hơn nhiều → nhiễu bị suy giảm mạnh')
print('  - Vùng tô đỏ nhạt: phần bị bộ lọc loại bỏ')

---
## 8. Tỷ số tín hiệu trên nhiễu (SNR)

### SNR (Signal-to-Noise Ratio) là gì?

$$SNR = 10 \cdot \log_{10}\left(\frac{P_{signal}}{P_{noise}}\right) \text{ dB}$$

Trong đó:
- **P_signal** = công suất tín hiệu sau lọc (phần giữ lại)
- **P_noise** = công suất phần bị loại (raw - filtered)

**Cách đọc:**
- SNR **cao** → tín hiệu sạch, ít nhiễu (tốt)
- SNR **thấp** → nhiều nhiễu (xấu)
- Ví dụ: SNR = 10 dB nghĩa là tín hiệu mạnh gấp 10 lần nhiễu

In [ ]:
snr = compute_snr(y_raw, y_filt)

print('=== TỶ SỐ TÍN HIỆU TRÊN NHIỄU ===')
print(f'  SNR sau lọc FIR: {snr:.2f} dB')
print()
print('  Giải thích:')
if snr > 15:
    print(f'  → SNR = {snr:.1f} dB > 15 dB: tín hiệu sạch, bộ lọc giữ lại phần lớn giọng nói')
elif snr > 5:
    print(f'  → SNR = {snr:.1f} dB: mức trung bình, bộ lọc loại được nhiễu nhưng cũng mất một ít tín hiệu')
else:
    print(f'  → SNR = {snr:.1f} dB < 5 dB: cần xem lại — có thể mất quá nhiều tín hiệu')

# Trực quan hóa: phần tín hiệu vs phần nhiễu
noise = y_raw - y_filt  # Phần bị loại bỏ

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(time, y_raw, linewidth=0.5, color='steelblue')
axes[0].set_title('Tín hiệu thô (Raw)')
axes[0].set_ylabel('Biên độ')
axes[0].grid(True, alpha=0.3)

axes[1].plot(time, y_filt, linewidth=0.5, color='darkorange')
axes[1].set_title('Tín hiệu giữ lại (Signal) — sau lọc FIR')
axes[1].set_ylabel('Biên độ')
axes[1].grid(True, alpha=0.3)

axes[2].plot(time, noise, linewidth=0.5, color='red', alpha=0.7)
axes[2].set_title('Phần bị loại bỏ (Noise) = Raw - Filtered')
axes[2].set_ylabel('Biên độ')
axes[2].set_xlabel('Thời gian (giây)')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Phân tách tín hiệu và nhiễu — SNR = {snr:.1f} dB', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/snr_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Pre-emphasis — Tăng cường tần số cao

### Pre-emphasis là gì?
Bộ lọc bậc 1 đơn giản:

$$y'[n] = y[n] - 0.97 \cdot y[n-1]$$

### Tại sao cần?
- Giọng nói tự nhiên có năng lượng **giảm ~6 dB/octave** khi tần số tăng
- Tần số cao (chứa phụ âm /s/, /f/, /t/) bị "chìm" so với tần số thấp (nguyên âm)
- Pre-emphasis **boost tần số cao** → phổ tần số **phẳng hơn**
- Kết quả: MFCC (Notebook 02) trích xuất được nhiều thông tin hơn từ vùng tần số cao

### Cách hoạt động:
- Mẫu hiện tại trừ đi 97% mẫu trước đó
- Phần thay đổi nhanh (tần số cao) → hiệu lớn → **được giữ/tăng cường**
- Phần thay đổi chậm (tần số thấp) → hiệu nhỏ → **bị giảm**

In [ ]:
# Áp dụng pre-emphasis lên tín hiệu đã lọc
y_emph = pre_emphasize(y_filt, alpha=0.97)

print(f'Trước pre-emphasis — RMS: {np.sqrt(np.mean(y_filt**2)):.4f}')
print(f'Sau pre-emphasis   — RMS: {np.sqrt(np.mean(y_emph**2)):.4f}')

# So sánh phổ tần số trước/sau pre-emphasis
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, sig, title, color in zip(
    axes,
    [y_filt, y_emph],
    ['Sau FIR (chưa pre-emphasis)', 'Sau FIR + Pre-emphasis'],
    ['darkorange', 'crimson']
):
    freqs = np.fft.rfftfreq(len(sig), d=1/SR)
    magnitude = np.abs(np.fft.rfft(sig))
    ax.semilogy(freqs, magnitude, linewidth=0.5, color=color)
    ax.axvline(300, color='red', linestyle='--', alpha=0.5)
    ax.axvline(3400, color='green', linestyle='--', alpha=0.5)
    ax.set_xlabel('Tần số (Hz)')
    ax.set_ylabel('Biên độ (log)')
    ax.set_title(title)
    ax.set_xlim(0, SR/2)
    ax.grid(True, alpha=0.3)

plt.suptitle('Hiệu ứng Pre-emphasis lên phổ tần số', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/preemphasis_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNhận xét:')
print('  - Phổ sau pre-emphasis "phẳng" hơn — tần số cao được tăng cường')
print('  - Điều này giúp MFCC (Notebook 02) trích xuất thông tin tốt hơn')
print('  - Pre-emphasis là bước thứ 2 trong Pipeline B: Audio → FIR → Pre-emphasis → MFCC')

---
## 10. So sánh nhiều speakers

Hãy xem phổ tần số của **4 speakers khác nhau** để hiểu tại sao bộ lọc giúp phân biệt giọng nói.

In [ ]:
# Load 1 file mẫu từ mỗi speaker
speakers = {
    'Cuong': '../data/raw/speaker_08/01.wav',
    'Quang': '../data/raw/speaker_09/01.wav',
    'Anne':  '../data/raw/speaker_10/01.wav',
    'Khoa':  '../data/raw/speaker_11/01.wav',
}
colors_spk = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, (name, path) in enumerate(speakers.items()):
    ax = axes[idx // 2][idx % 2]
    y_spk, _ = preprocess(path, sr=SR, target_len=TARGET_LEN)
    y_spk_filt = apply_filter(y_spk, coeffs)
    
    freqs = np.fft.rfftfreq(len(y_spk_filt), d=1/SR)
    magnitude = np.abs(np.fft.rfft(y_spk_filt))
    
    ax.semilogy(freqs, magnitude, linewidth=0.8, color=colors_spk[idx])
    ax.axvline(300, color='red', linestyle='--', alpha=0.4)
    ax.axvline(3400, color='green', linestyle='--', alpha=0.4)
    ax.set_title(f'{name} — Phổ tần số sau lọc FIR', fontsize=11)
    ax.set_xlabel('Tần số (Hz)')
    ax.set_ylabel('Biên độ (log)')
    ax.set_xlim(0, SR/2)
    ax.grid(True, alpha=0.3)

plt.suptitle('So sánh phổ tần số 4 speakers (sau lọc FIR)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/speakers_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

print('Nhận xét:')
print('  - Mỗi speaker có phổ tần số KHÁC NHAU → formants khác nhau')
print('  - Đây chính là thông tin để MFCC phân biệt giọng nói')
print('  - Bộ lọc FIR giúp tập trung vào dải tần chứa formants (300–3400 Hz)')

---
## Tổng kết Notebook 01

| Bước | Kỹ thuật | Kết quả |
|------|----------|--------|
| Tiền xử lý | Normalize + Trim + Pad | Audio chuẩn: 3s, 16kHz, [-1,1] |
| Bộ lọc FIR | Bandpass 300–3400 Hz, 101 tap, Hamming | Loại nhiễu ngoài dải giọng nói |
| FFT | Phổ tần số | Xác nhận năng lượng ngoài dải bị loại |
| STFT | Spectrogram | Thấy rõ formants trong dải 300–3400 Hz |
| PSD | Welch's method | Mật độ công suất giảm mạnh ngoài dải |
| SNR | Signal/Noise ratio | Đo lường hiệu quả lọc |
| Pre-emphasis | y'[n] = y[n] - 0.97*y[n-1] | Phổ phẳng hơn, tần số cao được tăng cường |

**Tiếp theo:** Notebook 02 — Trích xuất đặc trưng (MFCC, basic features) → Notebook 03 — Huấn luyện SVM